In [ ]:
# 加载环境变量
import os

import dotenv

dotenv.load_dotenv()
DEEPSEEK_API = os.getenv("DEEPSEEK_API")

In [ ]:
# 大模型
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="deepseek-v4-pro",
    base_url="https://api.deepseek.com/v1",
    api_key=DEEPSEEK_API,
    temperature=0.7,
    model_provider="openai",
)


In [ ]:
# backend
import os

from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend, StoreBackend

docs_root = os.path.abspath("./project_docs")


def composite_backend_factory(rt):
    return CompositeBackend(
        default=StateBackend(rt),
        routes={"/memories/": StoreBackend(rt), "/docs/": FilesystemBackend(root_dir=docs_root, virtual_mode=True)},
    )

In [ ]:
# 创建智能体
from deepagents import create_deep_agent
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

agent = create_deep_agent(
    model=model,
    backend=composite_backend_factory,
    store=store,
    system_prompt="""
    你有一个分层文件系统：
    - /workspace/ 下是短期工作区（临时）
    - /memories/ 是长期记忆（持久化）
    - /docs/ 下是本地项目文档 （只在当前机器存在）
    
    请合理使用这些路径完成任务
    """,
)